<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z341_RegLinealNorm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión lineal reciente con normalización

Igual que z330 pero normalizando la serie antes de ajustar la recta.

## Pipeline por producto
```
serie original
  → normalizar (max / L2 / index)
  → reg lineal sobre últimos N meses normalizados
  → pred normalizada
  → × escala
  → pred real
```

## ¿Por qué puede ayudar?

La reg lineal OLS minimiza el error cuadrático. Si la serie tiene un pico alto (ej: 500 tn un mes, 10 el resto), la recta se va a inclinar hacia ese pico porque el error cuadrático lo penaliza mucho.

Al normalizar por el máximo, ese pico vale 1.0 y el resto vale ~0.02. La recta ahora ajusta la **forma relativa** sin que el pico la domine.

## Variantes
Barremos las 3 normalizaciones × ventanas 3/6/12 meses → 9 combinaciones en backtesting + submit de las ganadoras.

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell
mkdir -p "/content/.drive/My Drive/labo3" /content/buckets
ln -sfn "/content/.drive/My Drive/labo3" /content/buckets/b1
mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json
mkdir -p /content/buckets/b1/datasets /content/datasets
descargar() {
  d="/content/buckets/b1/datasets/"
  u="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  if ! test -f "$d$1"; then wget "$u$1" -O "$d$1"; fi
  if ! test -f "/content/datasets/$1"; then cp "$d$1" "/content/datasets/$1"; fi
}
descargar sell-in.txt.gz
descargar product_id_apredecir201912.txt

In [ ]:
!pip install uv -q && uv pip install -q kaggle

In [ ]:
import os, shutil, itertools
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

PARAM = {
    'competencia':    'labo-iii-2026-rosario',
    'periodo_corte':  201910,
    'periodo_target': 201912,
    'horizonte':      2,
    'ventanas':       [3, 6, 12],
    'drive_path':     '/content/buckets/b1/exp/RegLinealNorm',
}

os.makedirs(PARAM['drive_path'], exist_ok=True)

dataset      = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator='\t')
tb_ventas    = dataset.group_by('product_id','periodo').agg(pl.col('tn').sum()).sort(['product_id','periodo'])
tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator='\t')
tb_ventas    = tb_ventas.join(tb_apredecir, on='product_id', how='inner').sort(['product_id','periodo'])
tb_train     = tb_ventas.filter(pl.col('periodo') <= PARAM['periodo_corte'])
tb_real      = (tb_ventas.filter(pl.col('periodo') == PARAM['periodo_target'])
                .select(['product_id','tn']).rename({'tn':'tn_real'}))
productos    = tb_apredecir['product_id'].to_list()
print(f'{len(productos)} productos')

# Normalizaciones y predictor

In [ ]:
def norm_max(serie):
    m = serie.max()
    return (serie / m, float(m)) if m > 0 else (serie.copy(), 1.0)

def norm_l2(serie):
    norma = float(np.sqrt((serie ** 2).sum()))
    return (serie / norma, norma) if norma > 0 else (serie.copy(), 1.0)

def norm_index(serie, n_base=3):
    positivos = serie[serie > 0]
    base = float(positivos[:n_base].mean()) if len(positivos) > 0 else 1.0
    return (serie / base, base) if base > 0 else (serie.copy(), 1.0)

NORMALIZACIONES = {'max': norm_max, 'l2': norm_l2, 'index': norm_index}


def pred_reg_norm(serie, ventana, horizonte, norm_fn):
    """
    1. Normaliza los últimos `ventana` meses
    2. Ajusta OLS sobre la serie normalizada
    3. Predice `horizonte` pasos adelante (normalizado)
    4. Desnormaliza
    """
    w = min(ventana, len(serie))
    if w < 2:
        return max(float(serie.mean()), 0.0)

    ventana_raw = serie[-w:]
    ventana_norm, escala = norm_fn(ventana_raw)

    x = np.arange(w).reshape(-1, 1)
    m = LinearRegression().fit(x, ventana_norm)
    pred_norm = float(m.predict([[w - 1 + horizonte]])[0])
    pred_norm = max(pred_norm, 0.0)

    return max(pred_norm * escala, 0.0)


print('OK')

# Backtesting — 3 normalizaciones × 3 ventanas

In [ ]:
reales = {row['product_id']: row['tn_real'] for row in tb_real.to_dicts()}

combos  = list(itertools.product(NORMALIZACIONES.keys(), PARAM['ventanas']))
rmse_bt = {}

# baseline: reg lineal sin normalizar
for v in PARAM['ventanas']:
    errores = []
    for pid in productos:
        serie = tb_train.filter(pl.col('product_id') == pid).sort('periodo')['tn'].to_numpy().astype(float)
        w = min(v, len(serie))
        if w < 2:
            pred = max(float(serie.mean()), 0.0)
        else:
            y = serie[-w:]
            x = np.arange(w).reshape(-1, 1)
            pred = max(float(LinearRegression().fit(x, y).predict([[w - 1 + PARAM['horizonte']]])[0]), 0.0)
        if pid in reales:
            errores.append((pred - reales[pid]) ** 2)
    rmse_bt[f'sin_norm_v{v}'] = float(np.sqrt(np.mean(errores)))

# variantes normalizadas
for nombre_norm, fn in NORMALIZACIONES.items():
    for v in PARAM['ventanas']:
        errores = []
        for pid in productos:
            serie = tb_train.filter(pl.col('product_id') == pid).sort('periodo')['tn'].to_numpy().astype(float)
            pred  = pred_reg_norm(serie, v, PARAM['horizonte'], fn)
            if pid in reales:
                errores.append((pred - reales[pid]) ** 2)
        rmse_bt[f'{nombre_norm}_v{v}'] = float(np.sqrt(np.mean(errores)))

print('RMSE backtesting 201912:')
print()
print('── Baseline sin normalizar ──')
for k in sorted(rmse_bt):
    if k.startswith('sin_norm'):
        print(f'  {k:20s}: {rmse_bt[k]:.4f}')
print()
print('── Con normalización ──')
for k, r in sorted(rmse_bt.items(), key=lambda x: x[1]):
    if not k.startswith('sin_norm'):
        delta = r - rmse_bt.get(f'sin_norm_v{k.split("v")[1]}', r)
        print(f'  {k:20s}: {r:.4f}  ({delta:+.4f} vs sin norm)')

# Visualización — efecto de la normalización en la recta

In [ ]:
# productos con mayor diferencia entre reg normal y reg norm_max (v6)
diffs = []
for pid in productos:
    serie = tb_train.filter(pl.col('product_id') == pid).sort('periodo')['tn'].to_numpy().astype(float)
    v = 6
    w = min(v, len(serie))
    if w < 2 or pid not in reales:
        continue
    y = serie[-w:]
    x = np.arange(w).reshape(-1, 1)
    p_sin  = max(float(LinearRegression().fit(x, y).predict([[w - 1 + 2]])[0]), 0.0)
    p_norm = pred_reg_norm(serie, v, 2, norm_max)
    diffs.append({'product_id': pid, 'p_sin': p_sin, 'p_norm': p_norm,
                  'real': reales[pid], 'diff': abs(p_sin - p_norm)})

tb_diff = pl.DataFrame(diffs).sort('diff', descending=True)
top6 = tb_diff.head(6)['product_id'].to_list()

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, pid in enumerate(top6):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos_  = serie_full['periodo'].to_list()
    tn_        = serie_full['tn'].to_numpy().astype(float)
    idx_corte  = next((j for j, p in enumerate(periodos_) if p > PARAM['periodo_corte']), len(periodos_))
    idx_target = next((j for j, p in enumerate(periodos_) if p == PARAM['periodo_target']), None)
    tn_tr = tn_[:idx_corte]

    row = tb_diff.filter(pl.col('product_id') == pid)
    p_sin  = float(row['p_sin'][0])
    p_norm = float(row['p_norm'][0])
    real   = float(row['real'][0])

    ax = axes[i]
    ax.plot(range(len(tn_tr)), tn_tr, 'o-', color='steelblue', markersize=3, linewidth=1.5)

    if idx_target is not None:
        t = idx_target
        ax.scatter([t], [real],   color='black',  s=90, zorder=6, label=f'real={real:.1f}')
        ax.scatter([t], [p_sin],  color='tomato', s=60, zorder=5, marker='D', label=f'sin norm={p_sin:.1f}')
        ax.scatter([t], [p_norm], color='green',  s=60, zorder=5, marker='D', label=f'norm max={p_norm:.1f}')

    ax.set_title(f'pid {pid}', fontsize=8)
    ax.legend(fontsize=6)

fig.suptitle('Casos donde norm_max cambia más la predicción vs reg sin normalizar\n'
             '(negro=real, rojo=sin norm, verde=con norm max)', fontsize=9)
plt.tight_layout()
plt.show()

# McNemar — significancia

In [ ]:
def mcnemar(err_a, err_b, nombre_a, nombre_b):
    n10 = (err_a < err_b).sum()
    n01 = (err_b < err_a).sum()
    if n10 + n01 == 0:
        return
    stat = (abs(n10 - n01) - 1)**2 / (n10 + n01)
    pval = 1 - chi2.cdf(stat, df=1)
    sig  = '** SIG **' if pval < 0.05 else 'no sig   '
    ganador = nombre_a if n10 > n01 else nombre_b
    print(f'  {nombre_a:20s} vs {nombre_b:20s}: {n10:3d}|{n01:3d}  p={pval:.4f}  {sig}  → {ganador}')

# precalcular errores por producto para cada variante
errores_por_variante = {}
for k in rmse_bt:
    errs = []
    partes = k.split('_v')
    v = int(partes[-1])
    nombre_norm = partes[0] if partes[0] != 'sin' else None
    fn = NORMALIZACIONES.get(nombre_norm) if nombre_norm else None

    for pid in productos:
        serie = tb_train.filter(pl.col('product_id') == pid).sort('periodo')['tn'].to_numpy().astype(float)
        if fn:
            pred = pred_reg_norm(serie, v, PARAM['horizonte'], fn)
        else:
            w = min(v, len(serie))
            if w < 2:
                pred = max(float(serie.mean()), 0.0)
            else:
                y = serie[-w:]
                x = np.arange(w).reshape(-1, 1)
                pred = max(float(LinearRegression().fit(x, y).predict([[w - 1 + PARAM['horizonte']]])[0]), 0.0)
        errs.append(abs(pred - reales.get(pid, pred)))
    errores_por_variante[k] = np.array(errs)

print('McNemar — normalizadas vs sin normalizar (misma ventana):')
print()
for v in PARAM['ventanas']:
    baseline = errores_por_variante[f'sin_norm_v{v}']
    for norm in NORMALIZACIONES:
        mcnemar(baseline, errores_por_variante[f'{norm}_v{v}'], f'sin_norm_v{v}', f'{norm}_v{v}')

# Submit — todas las variantes que superan al baseline

In [ ]:
def kaggle_submit(competencia, archivo, mensaje):
    os.system(f'kaggle competitions submit -c {competencia} -f {archivo} -m "{mensaje}"')

# baseline: mejor reg sin normalizar
rmse_baselines = {v: rmse_bt[f'sin_norm_v{v}'] for v in PARAM['ventanas']}

# variantes que superan al baseline de su ventana
a_submitear = [
    (k, r) for k, r in rmse_bt.items()
    if not k.startswith('sin_norm')
    and r < rmse_baselines[int(k.split('v')[-1])]
]
a_submitear.sort(key=lambda x: x[1])

print(f'{len(a_submitear)} variantes superan al baseline → submiteando...')
print()

for nombre_k, rmse_val in a_submitear:
    partes      = nombre_k.split('_v')
    v           = int(partes[-1])
    nombre_norm = partes[0]
    fn          = NORMALIZACIONES[nombre_norm]

    preds = []
    for pid in productos:
        serie = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')['tn'].to_numpy().astype(float)
        pred  = pred_reg_norm(serie, v, PARAM['horizonte'], fn)
        preds.append({'product_id': pid, 'tn': pred})

    archivo = f'reg_norm_{nombre_k}.csv'
    mensaje = f'RegLineal norm={nombre_norm} v={v} RMSE_bt={rmse_val:.4f}'

    pl.DataFrame(preds).write_csv(archivo)
    shutil.copy(archivo, f"{PARAM['drive_path']}/{archivo}")
    kaggle_submit(PARAM['competencia'], archivo, mensaje)
    print(f'  submitted: {archivo}  (RMSE_bt={rmse_val:.4f})')

if not a_submitear:
    print('Ninguna variante normalizada superó al baseline.')
    print('Submiteando la mejor en general:')
    mejor = min(rmse_bt.items(), key=lambda x: x[1])
    print(f'  {mejor[0]}: {mejor[1]:.4f}')